# 03 — Agentic Research Assistant
### AI Research Paper Intelligence System — Project 3

This notebook walks through the multi-agent layer built on top of the
semantic search engine from Project 2. Instead of one search query, you
give the system a **broad** topic, and a team of coordinated agents
autonomously plans multiple angles, retrieves papers, checks its own
coverage, and writes a literature-review report.

**Why "agentic"?** The key difference from a fixed pipeline is the
orchestrator's decision loop: after each round of searching, it checks
how many *unique* papers have actually been found and decides for itself
whether to keep searching (asking the planner for more angles) or move on
to writing the report. Nothing about the number of steps is hardcoded.

**No API key needed** — the planning step uses a small, free, local model
(`google/flan-t5-small`) rather than a paid LLM API.

## 1. Load the search engine

This is the same `PaperSearchEngine` from Project 2 — the agent system
uses it as a "tool" that the Retrieval Agent calls repeatedly.

In [ ]:
import sys
sys.path.append("../src")

from search_engine import PaperSearchEngine

engine = PaperSearchEngine()

## 2. Meet the agents

The system is composed of four specialized agents plus an orchestrator
that coordinates them:

| Agent | Responsibility |
|---|---|
| `PlannerAgent` | Breaks a broad topic into ~5 focused, searchable sub-queries |
| `RetrievalAgent` | Calls the search engine per sub-query, tracks unique papers found |
| `AnalystAgent` | Synthesizes findings per sub-topic, detects cross-cutting themes |
| `WriterAgent` | Compiles everything into a Markdown report |
| `ResearchAgent` (orchestrator) | Runs the full loop, deciding when enough ground has been covered |

In [ ]:
from agents.planner_agent import PlannerAgent
from agents.retrieval_agent import RetrievalAgent
from agents.analyst_agent import AnalystAgent
from agents.writer_agent import WriterAgent
from agents.orchestrator import ResearchAgent

## 3. Try the Planner Agent on its own

Let's see how it breaks down a broad topic before running the full
pipeline. It tries `flan-t5-small` first; if that model can't load (e.g.
no internet on first run), it automatically falls back to template-based
decomposition so the agent still works.

In [ ]:
planner = PlannerAgent(use_llm=True)

topic = "attention mechanisms in NLP"
sub_queries = planner.plan(topic, n=5)

print(f"Broad topic: {topic}\n")
print("Sub-queries generated:")
for q in sub_queries:
    print(" -", q)

## 4. Try the Retrieval Agent

The Retrieval Agent wraps the search engine and, importantly, **remembers**
every unique paper it has seen across multiple queries — this running
memory is what lets the orchestrator later measure "coverage." 

In [ ]:
retriever = RetrievalAgent(engine)

for q in sub_queries[:2]:
    new_papers = retriever.retrieve(q, k=4)
    print(f"Query: '{q}' -> {len(new_papers)} new paper(s)")

print("\nTotal unique papers gathered so far:", retriever.coverage())

## 5. Try the Analyst Agent

For each sub-topic, the Analyst Agent summarizes the top papers together
(a synthesis, not just one paper's summary) and extracts representative
keywords.

In [ ]:
analyst = AnalystAgent(engine)

# Re-run retrieval fresh for a clean demo
retriever.reset()
papers_for_first_query = retriever.retrieve(sub_queries[0], k=4)

result = analyst.synthesize_subtopic(sub_queries[0], papers_for_first_query)
print("Sub-query:", result["sub_query"])
print("Synthesis:", result["synthesis"])
print("Keywords:", result["keywords"])

## 6. Run the full autonomous pipeline

Now let's run the complete `ResearchAgent` orchestrator. It will:
1. Plan sub-queries
2. Retrieve papers for each, tracking unique coverage
3. **Decide** whether enough papers have been found (`min_coverage`) — if
   not, it asks the planner for more angles and searches again
4. Synthesize findings and detect cross-cutting themes
5. Write the final Markdown report

Watch the printed log below — this is the agent's reasoning trace.

In [ ]:
agent = ResearchAgent(engine, min_coverage=10, max_rounds=3)

report = agent.run("attention mechanisms in NLP", papers_per_query=4, verbose=True)

## 7. View the final report

In [ ]:
print(report)

## 8. Save the report to disk

In [ ]:
import os

os.makedirs("../data", exist_ok=True)
with open("../data/agent_report_attention_mechanisms.md", "w") as f:
    f.write(report)

print("Saved to ../data/agent_report_attention_mechanisms.md")

---
## Summary

We now have a complete agentic layer on top of the Project 2 search engine:

`broad topic → Planner → Retriever (with coverage tracking) → Orchestrator's
decision loop → Analyst → Writer → Markdown report`

This can also be run directly from the command line:

```bash
python ../src/run_agent.py "your broad topic here"
```

or from the Streamlit app's **🤖 Research Agent** tab:

```bash
streamlit run ../src/app.py
```